# Food-101: HBCC 2.5M vs ResNet-18 from scratch

**CoC/ImageNet recipe:** input 224x224, RandomResizedCrop 0.08--1.0, horizontal flip, RandAugment, Random Erasing 0.25, Mixup 0.8, CutMix 1.0, and label smoothing 0.1. Validation/test use deterministic resize + center crop.

Notebook chính cho pipeline source-based. Cả hai model khởi tạo ngẫu nhiên, dùng cùng split stratified, optimizer và CoC-style augmentation recipe. Test set chỉ được dùng đúng một lần sau khi chọn checkpoint tốt nhất bằng validation.

Trên Kaggle: bật Internet để `torchvision` tải Food-101 và Add Data toàn bộ repository này (hoặc sửa `PROJECT_ROOT`).

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
import pandas as pd
import matplotlib.pyplot as plt

# Có thể gán trực tiếp, ví dụ Path('/kaggle/input/lightweight-context-cluster').
PROJECT_ROOT = Path(os.environ.get('HBCC_PROJECT_ROOT', '')).expanduser() if os.environ.get('HBCC_PROJECT_ROOT') else None
if PROJECT_ROOT is None or not (PROJECT_ROOT / 'pyproject.toml').is_file():
    candidates = [Path.cwd(), Path.cwd().parent]
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.is_dir(): candidates += list(kaggle_input.iterdir()) + list(kaggle_input.glob('*/*'))
    PROJECT_ROOT = next((p for p in candidates if (p / 'pyproject.toml').is_file() and (p / 'tools' / 'run_food101_experiments.py').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Không tìm thấy source repo. Hãy đặt biến môi trường HBCC_PROJECT_ROOT hoặc sửa PROJECT_ROOT trong cell này.')
PROJECT_ROOT = PROJECT_ROOT.resolve()
IS_KAGGLE = Path('/kaggle/working').is_dir()
DATA_ROOT = Path('/kaggle/working/torchvision_data') if IS_KAGGLE else PROJECT_ROOT / 'data' / 'food101'
OUTPUT_ROOT = Path('/kaggle/working/food101_runs') if IS_KAGGLE else PROJECT_ROOT / 'runs' / 'food101'
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_ROOT   :', DATA_ROOT)
print('OUTPUT_ROOT :', OUTPUT_ROOT)


In [ ]:
# Cấu hình chạy chính. 60 epoch phù hợp giới hạn thời gian phổ biến của Kaggle.
MODELS = ['hbcc', 'resnet18']
EPOCHS = 60
DEVICE = 'auto'
SMOKE_TEST = False  # True: chỉ chạy 2 batch mỗi split trong 1 epoch

sys.path.insert(0, str(PROJECT_ROOT))
from lightweight_hbcc.models import build_model
from lightweight_hbcc.models.food101 import HBCC_FOOD101_BEST_CONFIG

hbcc = build_model({'model': {'name': 'hbcc_food101_best', 'num_classes': 101}})
resnet = build_model({'model': {'name': 'resnet18_224', 'num_classes': 101}})
print('HBCC parameters    :', f'{sum(p.numel() for p in hbcc.parameters()):,}')
print('ResNet-18 parameters:', f'{sum(p.numel() for p in resnet.parameters()):,}')
print('HBCC canonical config:')
display(pd.Series(HBCC_FOOD101_BEST_CONFIG, name='value').to_frame())
del hbcc, resnet


In [ ]:
runner = PROJECT_ROOT / 'tools' / 'run_food101_experiments.py'
command = [sys.executable, str(runner), '--models', *MODELS, '--output', str(OUTPUT_ROOT), '--data-root', str(DATA_ROOT), '--epochs', str(1 if SMOKE_TEST else EPOCHS), '--device', DEVICE, '--no-progress']
if SMOKE_TEST:
    command += ['--limit-train-batches', '2', '--limit-val-batches', '2', '--limit-test-batches', '2']
print('Running:', ' '.join(command))
subprocess.run(command, cwd=PROJECT_ROOT, check=True)


## Kết quả
Bảng cuối dùng checkpoint có validation accuracy tốt nhất. Test accuracy không được dùng để chọn model hay epoch.

In [ ]:
run_names = {'hbcc': 'food101_hbcc_2p5m_coc_recipe_seed42', 'resnet18': 'food101_resnet18_scratch_coc_recipe_seed42'}
histories = []
for model_key in MODELS:
    metrics_path = OUTPUT_ROOT / run_names[model_key] / 'metrics.jsonl'
    records = [json.loads(line) for line in metrics_path.read_text(encoding='utf-8').splitlines() if line.strip()]
    frame = pd.DataFrame([row for row in records if 'val_acc1' in row])
    frame['model'] = model_key
    frame['epoch_display'] = frame['epoch'] + 1
    histories.append(frame)
history = pd.concat(histories, ignore_index=True)
comparison = pd.read_csv(OUTPUT_ROOT / 'comparison.csv').sort_values('test_acc1', ascending=False)
display(comparison)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for model_key, group in history.groupby('model'):
    axes[0].plot(group.epoch_display, group.train_acc1, label=model_key)
    axes[1].plot(group.epoch_display, group.val_acc1, label=model_key)
    axes[2].plot(group.epoch_display, group.val_loss, label=model_key)
axes[0].set_title('Train soft-target accuracy'); axes[1].set_title('Validation accuracy'); axes[2].set_title('Validation loss')
for axis in axes:
    axis.set_xlabel('Epoch'); axis.grid(True); axis.legend()
axes[0].set_ylabel('%'); axes[1].set_ylabel('%'); axes[2].set_ylabel('CE loss')
plt.tight_layout(); plt.show()
